# Amparo -- M3 · S10 · Extra: optimizar el prompt con DSPy

Hasta aqui el prompt de generacion del RAG (`prompt_template`: instruccion +
valvula de escape) se escribio a mano. Con una metrica confiable, DSPy puede
buscar un prompt mejor: prueba instrucciones y ejemplos (few-shot), mide cada
intento y se queda con lo que mas puntua.

**Que se optimiza:** el prompt de generacion del RAG de una pasada. El retrieval
queda fijo (configuracion C, hybrid + rerank): los contextos se calculan una vez y
se reusan, asi que cualquier cambio en el puntaje es atribuible al prompt.

**Contra que metrica:** `dspy_prompt.metrica_amparo`, programatica (sin juez LLM,
para no gastar el cupo de Groq en cientos de intentos). Mide el principio de
Amparo, ser honesto con sus fuentes: en gold, responder citando solo articulos
del contexto (0.5 si responde sin citar); en adversariales, ser prudente. No mide
si la respuesta es juridicamente correcta: eso lo mide RAGAS en la evaluacion final.

**Con que datos (sin tocar el examen):**

| Conjunto | Gold | Adversariales | Uso |
|---|---|---|---|
| train | 40 del dataset de M1 | 6 nuevos | DSPy arma instrucciones y demos |
| dev | 20 del dataset de M1 | 4 nuevos | elegir entre base / BootstrapFewShot / MIPROv2 |
| test | eval set (50) | eval set (6) | una sola vez al final, comparable con las otras rutas |

**Fases:** datos y contextos → modelo en Ollama → base → `BootstrapFewShot` →
`MIPROv2` (light) → elegir y exportar → test con el generador de HF → RAGAS y W&B.

**Requiere GPU** (T4 o superior), el indice de `m3_s07_rag_ingenuo.ipynb` en Drive
y los secretos de Colab `GROQ_API_KEY` y `WANDB_API_KEY` (fase 9). Decisiones en
`docs/m3_decisiones_rag.md`, seccion 24.

In [ ]:
# Rama del repo a clonar. Cambiala si el codigo del RAG avanzado todavia no esta
# en main (por ejemplo, mientras vive en una rama de PR).
RAMA = "RAGAS"

import os

if not os.path.isdir("Amparo"):
    !git clone -b {RAMA} https://github.com/TomasPosada0626/Amparo.git
else:
    !cd Amparo && git fetch origin && git checkout {RAMA} && git pull

%cd Amparo

In [ ]:
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

REQUERIDOS = [
    "tools/rag/hybrid.py",
    "tools/rag/rerank.py",
    "tools/rag/tools.py",
    "tools/rag/agentico.py",
    "tools/rag/dspy_prompt.py",
    "tools/rag/pipeline.py",
    "data/eval_set.json",
]
faltan = [r for r in REQUERIDOS if not os.path.exists(r)]
if faltan:
    raise SystemExit(
        f"Falta en la rama '{RAMA}': {faltan}\n\n"
        "Este notebook corre contra el repo REMOTO. Si el codigo de DSPy "
        "todavia esta solo en tu maquina, commitealo y pusheralo, o cambia RAMA."
    )
print("Repo OK.")

In [ ]:
# Dependencias livianas del repo (incluye faiss-cpu y rank_bm25, el BM25 de la
# hybrid search).
!pip install -q -r requirements.txt

In [ ]:
# Stack pesado: se instala aqui y no en requirements.txt, para no reemplazar el
# build de PyTorch con CUDA que Colab ya trae. sentence-transformers trae el
# cross-encoder del reranking; transformers alcanza para e5 y la generacion;
# peft/bitsandbytes solo si se genera con el adaptador LoRA.
!pip install -q -U transformers sentence-transformers peft bitsandbytes accelerate

# DSPy fijado a la version con la que se probo el codigo (optuna: lo pide MIPROv2).
!pip install -q "dspy[optuna]==3.4.0"

## Fase 1 -- Cargar el indice y el BM25

Igual que en S08 y S10: el indice denso se copia de Drive y el BM25 se construye
una vez en memoria.

In [ ]:
from google.colab import drive
import pathlib, shutil
from tools.rag import config

drive.mount("/content/drive")
origen = pathlib.Path(config.DRIVE_ROOT) / "rag"
config.ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
for nombre in ("rag_index.faiss", "rag_index_metadata.jsonl"):
    shutil.copy(origen / nombre, config.ARTIFACTS_DIR / nombre)
print("Indice copiado de Drive a artifacts/.")

In [ ]:
from tools.rag import pipeline
from tools.rag.hybrid import BM25Index

store = pipeline.load_index()
bm25 = BM25Index(store.metadata)   # una sola vez; se reusa en cada consulta
print(f"Indice denso: {len(store)} chunks | BM25: {len(bm25)} chunks")

## Fase 2 -- Datos y contextos

`dividir_datos` arma train y dev (determinista, semilla 42) sin ninguna pregunta
del eval set. Para cada caso se recupera el contexto con la configuracion C y se
guarda en Drive: si la sesion se cae, no hay que volver a recuperar.

In [ ]:
import json, pathlib
from tools.rag import dspy_prompt as dp
from tools.rag.retrieve import retrieve
from tools.evaluation import eval_set as eval_set_mod

DSPY_DIR = pathlib.Path(config.DRIVE_ROOT) / "rag" / "dspy"
DSPY_DIR.mkdir(parents=True, exist_ok=True)
cache = DSPY_DIR / "datos_con_contexto.json"

if cache.exists():
    datos = json.loads(cache.read_text(encoding="utf-8"))
    print("Datos cargados de Drive.")
else:
    train, dev = dp.dividir_datos(dp.cargar_dataset_m1(), eval_set_mod.load_eval_set())
    def con_ctx(caso):
        return dp.con_contexto(caso, retrieve(caso["pregunta"], store, bm25=bm25, use_hybrid=True, use_rerank=True))
    datos = {"train": [con_ctx(c) for c in train], "dev": [con_ctx(c) for c in dev]}
    cache.write_text(json.dumps(datos, ensure_ascii=False, indent=1), encoding="utf-8")

for nombre, casos in datos.items():
    sin_ctx = sum(1 for c in casos if not c["articulos"])
    print(f"{nombre}: {len(casos)} casos ({sum(c['tipo'] == 'gold' for c in casos)} gold, "
          f"{sum(c['tipo'] == 'adversarial' for c in casos)} adversariales) | sin contexto: {sin_ctx}")

## Fase 3 -- El modelo en Ollama

DSPy habla con el modelo por una API compatible con OpenAI; Ollama la expone.
Se usa `qwen2.5:7b`: misma familia y tamaño que el generador de HF, con otra
cuantizacion (GGUF Q4 en vez de bitsandbytes 4-bit). Por eso la optimizacion
corre aqui, pero el **test final corre con el generador de HF** (fase 8): la cifra
que se reporta sale del mismo modelo que las demas rutas.

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
ollama = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!ollama pull qwen2.5:7b

In [ ]:
import dspy

MODELO_OLLAMA = "qwen2.5:7b"
lm = dspy.LM(f"ollama_chat/{MODELO_OLLAMA}", api_base="http://localhost:11434", api_key="",
             temperature=0.0, max_tokens=config.MAX_NEW_TOKENS_GENERATION)
dspy.configure(lm=lm)

trainset = dp.a_ejemplos_dspy(datos["train"])
devset = dp.a_ejemplos_dspy(datos["dev"])
evaluar = dspy.Evaluate(devset=devset, metric=dp.metrica_amparo, num_threads=1, display_progress=True)
print(f"LM: {lm.model} | train {len(trainset)} | dev {len(devset)}")

## Fase 4 -- Punto de partida: el prompt escrito a mano

Mismas reglas de `prompt_template` (instruccion + valvula de escape), sin
ejemplos, medidas con la misma metrica sobre dev. Es la vara contra la que se
compara todo lo demas.

In [ ]:
puntajes, programas = {}, {}

programas["base"] = dp.construir_programa()
puntajes["base"] = evaluar(programas["base"]).score
print("Base (dev):", puntajes["base"])

## Fase 5 -- BootstrapFewShot: elegir ejemplos

Corre el programa sobre train, se queda con las respuestas que sacan 1.0 en la
metrica y las usa como ejemplos (hasta 3) en el prompt. No cambia la instruccion.

In [ ]:
from dspy.teleprompt import BootstrapFewShot

bfs = BootstrapFewShot(metric=dp.metrica_amparo, max_bootstrapped_demos=dp.MAX_DEMOS, max_labeled_demos=dp.MAX_DEMOS)
programas["bootstrap"] = bfs.compile(dp.construir_programa(), trainset=trainset)
puntajes["bootstrap"] = evaluar(programas["bootstrap"]).score
print("BootstrapFewShot (dev):", puntajes["bootstrap"], "| demos:", len(programas["bootstrap"].demos))

json.dump(dp.exportar_prompt(programas["bootstrap"], origen="BootstrapFewShot", puntajes=puntajes),
          open(DSPY_DIR / "prompt_bootstrap.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

## Fase 6 -- MIPROv2 (light): optimizar tambien la instruccion

Propone varias instrucciones candidatas (a partir del programa, de los datos y de
los demos), las combina con conjuntos de ejemplos y busca la mejor combinacion
con optimizacion bayesiana, midiendo en dev. Es la parte que va mas alla del
ejemplo de clase. En modo `light` y con este dev, del orden de una hora en T4.

In [ ]:
from dspy.teleprompt import MIPROv2

mipro = MIPROv2(metric=dp.metrica_amparo, auto="light", max_bootstrapped_demos=dp.MAX_DEMOS,
                max_labeled_demos=dp.MAX_DEMOS, num_threads=1, seed=dp.SEMILLA)
programas["mipro"] = mipro.compile(dp.construir_programa(), trainset=trainset, valset=devset, minibatch=False)
puntajes["mipro"] = evaluar(programas["mipro"]).score
print("MIPROv2 (dev):", puntajes["mipro"])

json.dump(dp.exportar_prompt(programas["mipro"], origen="MIPROv2-light", puntajes=puntajes),
          open(DSPY_DIR / "prompt_mipro.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

## Fase 7 -- Elegir y exportar

Gana el mayor puntaje en dev; ante empate, el mas simple (base, luego
BootstrapFewShot). El ganador se exporta como JSON (instrucciones + demos) a
Drive. **Para versionarlo**, copien ese archivo al repo como
`results/m3_dspy_prompt.json` y comitéenlo: asi queda trazable que prompt se
evaluo.

In [ ]:
orden = ["base", "bootstrap", "mipro"]
ganador = max(orden, key=lambda k: (puntajes[k], -orden.index(k)))
print("Puntajes en dev:", puntajes, "-> ganador:", ganador)

prompt_final = dp.exportar_prompt(programas[ganador], origen=ganador, puntajes=puntajes)
ruta_prompt = DSPY_DIR / "prompt_optimizado.json"
json.dump(prompt_final, open(ruta_prompt, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("Guardado en:", ruta_prompt)

print("\n=== INSTRUCCIONES ===\n" + prompt_final["instrucciones"])
for i, d in enumerate(prompt_final["demos"], start=1):
    print(f"\n=== DEMO {i} ===\nPREGUNTA: {d['pregunta']}\nRESPUESTA: {d['respuesta'][:400]}")

## Fase 8 -- Test final sobre el eval set, con el generador de HF

Se apaga Ollama (libera la GPU) y se carga Qwen2.5-7B de HF, el mismo generador
de las rutas de S10. El prompt ganador se pasa a `pipeline.answer_query` con
`prompt_optimizado=...`; el retrieval es el mismo (configuracion C). La salida
queda marcada como `una_pasada_dspy`.

Si la corrida de S10 ya existe en Drive, se compara con su ruta `una_pasada`
(el prompt original) con la misma metrica; si no, se corre tambien el prompt
original aqui.

In [ ]:
ollama.terminate(); subprocess.run(["pkill", "-f", "ollama"], check=False)
import gc, torch
gc.collect(); torch.cuda.empty_cache()

USE_LORA = False
etiqueta = "lora" if USE_LORA else "base"
model_bundle = pipeline.load_model(use_lora=USE_LORA)
registros = eval_set_mod.load_eval_set()
corridas_dir = pathlib.Path(config.DRIVE_ROOT) / "rag" / "corridas_s10"
corridas_dir.mkdir(parents=True, exist_ok=True)

def correr(prompt=None):
    records, t0 = [], time.perf_counter()
    for reg in registros:
        r = pipeline.answer_query(reg["messages"][1]["content"], store, use_lora=USE_LORA, bm25=bm25,
                                  model_bundle=model_bundle, use_hybrid=True, use_rerank=True,
                                  prompt_optimizado=prompt)
        records.append(pipeline.to_eval_record(r, reg))
    return records, (time.perf_counter() - t0) / len(registros)

corridas, latencias = {}, {}
corridas["una_pasada_dspy"], latencias["una_pasada_dspy"] = correr(prompt_final)

archivo_base = corridas_dir / f"eval_records_una_pasada_{etiqueta}.json"
if archivo_base.exists():
    corridas["una_pasada"] = json.loads(archivo_base.read_text(encoding="utf-8"))
    lat = corridas_dir / f"latencias_{etiqueta}.json"
    latencias["una_pasada"] = json.loads(lat.read_text())["una_pasada"] if lat.exists() else None
else:
    corridas["una_pasada"], latencias["una_pasada"] = correr(None)

json.dump(corridas["una_pasada_dspy"], open(corridas_dir / f"eval_records_una_pasada_dspy_{etiqueta}.json", "w",
          encoding="utf-8"), ensure_ascii=False, indent=2)

print(f"{'ruta':<17}{'metrica test':>13}{'gold':>7}{'adv':>7}{'s/cons':>8}")
for ruta, recs in corridas.items():
    p = [dp.puntaje_de_record(r) for r in recs]
    g = [dp.puntaje_de_record(r) for r in recs if r["tipo"] == "gold"]
    a = [dp.puntaje_de_record(r) for r in recs if r["tipo"] == "adversarial"]
    lat = latencias.get(ruta)
    print(f"{ruta:<17}{sum(p)/len(p):>13.3f}{sum(g)/len(g):>7.3f}{sum(a)/len(a):>7.3f}"
          f"{(f'{lat:.1f}' if lat else '-'):>8}")

## Fase 9 -- RAGAS (juez Groq) y registro en W&B

La metrica de DSPy mide honestidad con las fuentes; RAGAS mide ademas si la
respuesta se apoya en el contexto y contesta lo que se pregunto. El prompt
optimizado se adopta solo si no empeora RAGAS: optimizar una metrica y
degradar las demas no es una mejora. Mismo juez, checkpoint y proyecto de W&B
que en S10.

In [ ]:
import os, datetime, wandb
from google.colab import userdata
from tools.evaluation import ragas_metrics as rm, tracking, external_judge

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
juez, embed = rm.juez_groq(), rm.embedder_e5()

filas_ragas = {ruta: rm.evaluar_corrida(recs, juez=juez, embed=embed,
                                        checkpoint_path=corridas_dir / f"ragas_checkpoint_{ruta}_{etiqueta}.jsonl")
               for ruta, recs in corridas.items()}
resumenes = rm.resumen([f for fs in filas_ragas.values() for f in fs])
escapes = rm.tasas_de_escape([r for rs in corridas.values() for r in rs])
for ruta in corridas:
    r, e = resumenes.get(ruta, {}), escapes.get(ruta, {})
    print(f"{ruta:<17} faith={r.get('faithfulness')} c.prec={r.get('context_precision')} "
          f"c.rec={r.get('context_recall')} a.rel={r.get('answer_relevancy')} "
          f"prud.adv={e.get('prudencia_en_adversariales')} cita!gold={e.get('citas_no_respaldadas_en_gold')}")

try:
    wandb_key = userdata.get("WANDB_API_KEY")
except Exception:
    wandb_key = None
print("W&B en modo:", tracking.modo_de_ejecucion(wandb_key))
WANDB_ENTITY, WANDB_PROJECT = "tomasposada67-universidad-eafit", "amparo-rag"
config_run = {"generador": "Qwen/Qwen2.5-7B-Instruct", "use_lora": USE_LORA, "retrieval": "hybrid + rerank (config C)",
              "juez_ragas": external_judge.GROQ_JUDGE_MODEL, "dspy_version": dp.VERSION_DSPY,
              "optimizador_ganador": ganador, "puntajes_dev": puntajes,
              "instrucciones": prompt_final["instrucciones"][:1000], "n_demos": len(prompt_final["demos"])}
url = tracking.registrar_ruta(
    wandb, sistema="una_pasada_dspy", resumen_ragas=resumenes.get("una_pasada_dspy", {}),
    escape=escapes.get("una_pasada_dspy", {}), latencia_s=latencias.get("una_pasada_dspy"),
    filas_ragas=filas_ragas["una_pasada_dspy"], records=corridas["una_pasada_dspy"], config=config_run,
    entity=WANDB_ENTITY, project=WANDB_PROJECT, grupo=f"dspy-{etiqueta}-{datetime.date.today().isoformat()}",
)
print("W&B:", url or "(offline)")

---
### Como leer el resultado y que hacer

- **El prompt optimizado gana** si mejora la metrica de test y no empeora RAGAS
  (sobre todo faithfulness y answer relevancy) ni la prudencia en adversariales.
  Entonces: copiar `prompt_optimizado.json` a `results/m3_dspy_prompt.json`,
  comitearlo y usarlo con `pipeline.answer_query(prompt_optimizado=dp.cargar_prompt(...))`.
- **No gana** (o gana en dev y no en test): se reporta igual. Es evidencia de que
  el prompt escrito a mano ya estaba cerca del optimo para esta metrica, o de que
  dev es chico; no se adopta.
- La latencia importa: los demos alargan el prompt de cada consulta.

*Amparo · M3 · S10 · Extra DSPy.*